# Layer 2 — NLP: FinBERT Document Sentiment Pipeline

Ports the real pipeline from your `RM_Part_2` notebook: download RBI/Budget
policy documents, extract clean text, chunk by tokens, score with FinBERT,
and merge into your master dataset — pointed at the new backend folders.

**Output:**
- `backend/policy_docs/*.pdf` / `*.html` — downloaded documents
- `backend/policy_docs/finbert_sentiment_scores.csv` — per-document scores
- `backend/data/extended_2000_2026_with_sentiment.csv` — merged dataset (new file, doesn't touch your locked Layer 1 training data)

Run top to bottom. Takes a while — downloading ~37 documents + loading FinBERT (~400MB, first run only).

In [ ]:
# CELL 1 — Setup
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install",
                "transformers", "torch", "pdfplumber", "requests",
                "beautifulsoup4", "tqdm", "tf-keras", "--quiet"])

import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import time
import re
import numpy as np
import pandas as pd
import pdfplumber
from pathlib import Path
from bs4 import BeautifulSoup

BACKEND_DIR = Path(r"C:\final project\backend")
DATA_DIR    = BACKEND_DIR / "data"
POLICY_DIR  = BACKEND_DIR / "policy_docs"
POLICY_DIR.mkdir(parents=True, exist_ok=True)

print(f"Documents will be saved to: {POLICY_DIR}")

## Step 1 — Download policy documents (RBI MPC, Budget speeches, Economic Survey, RBI Governor pressers)

In [ ]:
# CELL 2 — Document URLs (RBI MPC statements + RBI Governor pressers, HTML)
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0"}

HTML_DOCS = {
    "mpc_2016_10": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=38248",
    "mpc_2017_02": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=39463",
    "mpc_2017_08": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=41364",
    "mpc_2018_02": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=43352",
    "mpc_2018_08": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=44937",
    "mpc_2019_02": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=46226",
    "mpc_2019_08": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=47843",
    "mpc_2020_03": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=49581",
    "mpc_2020_05": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=49843",
    "mpc_2020_08": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=50348",
    "mpc_2021_02": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=51058",
    "mpc_2021_08": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=52100",
    "mpc_2022_02": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=53238",
    "mpc_2022_08": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=54327",
    "mpc_2023_02": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=55364",
    "mpc_2023_08": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=56392",
    "mpc_2024_02": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=57423",
    "mpc_2024_08": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=58467",
    "mpc_2025_02": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=59012",
    "rbi_gov_2020_04": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=49679",
    "rbi_gov_2021_04": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=51380",
    "rbi_gov_2022_04": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=53571",
    "rbi_gov_2023_04": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=55596",
    "rbi_gov_2024_04": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=57648",
    "rbi_gov_2025_04": "https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=59234",
}

# Budget speeches + Economic Survey (PDF) — India Budget site changes URL
# patterns yearly, so we try several known patterns per document.
PDF_DOC_CANDIDATES = {
    "budget_2016": ["https://static.pib.gov.in/WriteReadData/specificdocs/documents/2016/feb/doc20162291.pdf"],
    "budget_2017": ["https://www.indiabudget.gov.in/budget2017-18/doc/Budget_Speech.pdf"],
    "budget_2018": ["https://www.indiabudget.gov.in/budget2018-19/doc/Budget_Speech.pdf"],
    "budget_2019": ["https://www.indiabudget.gov.in/budget2019-20/doc/Budget_Speech.pdf"],
    "budget_2020": ["https://www.indiabudget.gov.in/budget2020-21/doc/Budget_Speech.pdf"],
    "budget_2021": ["https://www.indiabudget.gov.in/budget2021-22/doc/Budget_Speech.pdf"],
    "budget_2022": ["https://www.indiabudget.gov.in/budget2022-23/doc/Budget_Speech.pdf"],
    "budget_2023": ["https://www.indiabudget.gov.in/budget2023-24/doc/Budget_Speech.pdf"],
    "budget_2024": ["https://www.indiabudget.gov.in/budget2024-25/doc/Budget_Speech.pdf"],
    "budget_2025": ["https://www.indiabudget.gov.in/budget2025-26/doc/Budget_Speech.pdf"],
    "econ_survey_2018": ["https://www.indiabudget.gov.in/budget2018-2019/es2017-18/echap01.pdf"],
    "econ_survey_2019": ["https://www.indiabudget.gov.in/budget2019-2020/es2018-19/echap01.pdf"],
    "econ_survey_2020": ["https://www.indiabudget.gov.in/budget2020-2021/es2019-20/echap01.pdf"],
    "econ_survey_2021": ["https://www.indiabudget.gov.in/budget2021-2022/es2020-21/echap01.pdf"],
    "econ_survey_2022": ["https://www.indiabudget.gov.in/budget2022-2023/es2021-22/echap01.pdf"],
    "econ_survey_2023": ["https://www.indiabudget.gov.in/budget2023-2024/es2022-23/echap01.pdf"],
    "econ_survey_2024": ["https://www.indiabudget.gov.in/budget2024-2025/es2023-24/echap01.pdf"],
    "econ_survey_2025": ["https://www.indiabudget.gov.in/budget2025-2026/es2024-25/echap01.pdf"],
}

print(f"Total documents to fetch: {len(HTML_DOCS) + len(PDF_DOC_CANDIDATES)}")

In [ ]:
# CELL 3 — Download all documents (skips any already downloaded)
def download_file(name, url, ext):
    out_path = POLICY_DIR / f"{name}.{ext}"
    if out_path.exists():
        print(f"  already have: {name}")
        return True
    try:
        import requests
        r = requests.get(url, headers=HEADERS, timeout=30)
        if r.status_code == 200 and len(r.content) > 3000:
            out_path.write_bytes(r.content)
            print(f"  downloaded  : {name} ({len(r.content)//1024} KB)")
            return True
        print(f"  failed ({r.status_code}): {name}")
        return False
    except Exception as e:
        print(f"  error: {name} - {str(e)[:60]}")
        return False

ok, fail = 0, 0
for name, url in HTML_DOCS.items():
    if download_file(name, url, "html"):
        ok += 1
    else:
        fail += 1
    time.sleep(0.5)

for name, urls in PDF_DOC_CANDIDATES.items():
    success = False
    for url in urls:
        if download_file(name, url, "pdf"):
            success = True
            break
        time.sleep(0.5)
    ok += success
    fail += (not success)

print(f"\nDownloaded: {ok} | Failed: {fail}")
all_files = sorted(list(POLICY_DIR.glob('*.pdf')) + list(POLICY_DIR.glob('*.html')))
print(f"Total documents ready: {len(all_files)}")

## Step 2 — Load FinBERT

In [ ]:
# CELL 4 — Load FinBERT (downloads weights first run only, ~400MB)
import torch
from transformers import BertTokenizer, BertForSequenceClassification, pipeline

print(f"PyTorch: {torch.__version__}")
print("Loading FinBERT weights...")

_tok = BertTokenizer.from_pretrained("ProsusAI/finbert")
_model = BertForSequenceClassification.from_pretrained("ProsusAI/finbert", ignore_mismatched_sizes=True)
_model.eval()

finbert = pipeline("text-classification", model=_model, tokenizer=_tok,
                    top_k=None, framework="pt", device=-1)

test = finbert("India economy faces severe recession risk amid collapsing output")
print(f"FinBERT loaded and working: {test[0]}")

## Step 3 — Extract text, chunk by tokens, score with FinBERT

In [ ]:
# CELL 5 — Extraction + chunking + scoring functions
def extract_pdf_clean(path, max_pages=20):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages[:max_pages]:
            t = page.extract_text()
            if t:
                text += t + " "
    return text.strip()

def extract_html_clean(path):
    soup = BeautifulSoup(Path(path).read_bytes(), "html.parser")
    paras = [p.get_text(strip=True) for p in soup.find_all("p") if len(p.get_text(strip=True)) > 30]
    if len(paras) < 3:
        # fall back to <td> — some RBI pages use tables, not paragraphs
        paras = [td.get_text(strip=True) for td in soup.find_all("td") if len(td.get_text(strip=True)) > 30]
    return " ".join(paras).strip()

def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\x00-\x7F]", " ", text)
    return text.strip()

def score_document(text, max_tokens=400, max_chunks=20):
    """Chunk by TOKENS (not words) to respect FinBERT's 512-token limit."""
    ids = _tok.encode(text, add_special_tokens=False)
    if len(ids) < 20:
        return None
    pos, neg, neu = [], [], []
    for i in range(0, min(len(ids), max_tokens * max_chunks), max_tokens):
        chunk_ids = ids[i:i + max_tokens]
        chunk_text = _tok.decode(chunk_ids, skip_special_tokens=True)
        if len(chunk_text.strip()) < 30:
            continue
        try:
            result = finbert(chunk_text, truncation=True, max_length=512)[0]
            s = {r["label"]: r["score"] for r in result}
            pos.append(s.get("positive", 0))
            neg.append(s.get("negative", 0))
            neu.append(s.get("neutral", 0))
        except Exception:
            continue
    if not pos:
        return None
    return {
        "positive": round(float(np.mean(pos)), 4),
        "negative": round(float(np.mean(neg)), 4),
        "neutral":  round(float(np.mean(neu)), 4),
        "composite": round(float(np.mean(pos)) - float(np.mean(neg)), 4),
        "n_chunks": len(pos),
    }

def parse_doc_date(name):
    parts = name.split("_")
    nums = [p for p in parts if p.isdigit()]
    if not nums:
        return None
    year = int(nums[0])
    if "mpc" in name or "rbi" in name:
        month = int(nums[1]) if len(nums) > 1 else 2
    elif "budget" in name:
        month = 2
    else:
        month = 1
    return pd.Timestamp(year=year, month=month, day=1)

print("Extraction, chunking, and scoring functions ready.")

In [ ]:
# CELL 6 — Score every document
records = []
all_files = sorted(list(POLICY_DIR.glob("*.pdf")) + list(POLICY_DIR.glob("*.html")))

print(f"Scoring {len(all_files)} documents...\n")
print(f"  {'Document':<32} {'Score':>7}  {'Pos':>6}  {'Neg':>6}  {'Chunks':>6}")
print("  " + "-" * 62)

for fpath in all_files:
    name = fpath.stem
    try:
        text = extract_pdf_clean(fpath) if fpath.suffix == ".pdf" else extract_html_clean(fpath)
        text = clean_text(text)
    except Exception as e:
        print(f"  [extract error] {name}: {e}")
        continue

    if len(text.split()) < 50:
        print(f"  [too short] {name} ({len(text.split())} words)")
        continue

    scores = score_document(text)
    if scores is None:
        print(f"  [no score] {name}")
        continue

    date = parse_doc_date(name)
    if date is None:
        print(f"  [no date] {name}")
        continue

    doc_type = ("budget" if "budget" in name else
                "econ_survey" if "econ" in name else
                "mpc" if "mpc" in name else "rbi_governor")

    records.append({
        "date": date, "doc_name": name, "doc_type": doc_type,
        "positive": scores["positive"], "negative": scores["negative"],
        "neutral": scores["neutral"], "finbert_score": scores["composite"],
        "n_chunks": scores["n_chunks"],
    })

    s = scores["composite"]
    tag = "positive" if s > 0.1 else "negative" if s < -0.1 else "neutral"
    print(f"  [{tag:<8}] {name:<32} {s:>+7.3f}  {scores['positive']:>6.3f}  {scores['negative']:>6.3f}  {scores['n_chunks']:>6}")

sentiment_df = pd.DataFrame(records).sort_values("date").reset_index(drop=True)
out_csv = POLICY_DIR / "finbert_sentiment_scores.csv"
sentiment_df.to_csv(out_csv, index=False)

print(f"\nScored {len(sentiment_df)}/{len(all_files)} documents")
print(f"Saved to: {out_csv}")
print("\nBy document type:")
print(sentiment_df.groupby("doc_type")["finbert_score"].agg(["mean", "min", "max", "count"]).round(3))

## Step 4 — Build monthly weighted sentiment composite and merge into your dataset

In [ ]:
# CELL 7 — Monthly composite + merge into the 302-month extended dataset
extended_path = DATA_DIR / "extended_2000_2026.csv"
if not extended_path.exists():
    raise FileNotFoundError(
        f"{extended_path} not found. Run the Layer 1 notebook first "
        "(Build_Full_System_2000.ipynb) to create it."
    )

master = pd.read_csv(extended_path, parse_dates=["date"])

date_range = pd.date_range(start=master["date"].min(), end=master["date"].max(), freq="MS")
monthly = pd.DataFrame({"date": date_range})

for doc_type in ["budget", "econ_survey", "mpc", "rbi_governor"]:
    sub = (sentiment_df[sentiment_df["doc_type"] == doc_type][["date", "finbert_score"]]
           .rename(columns={"finbert_score": f"sent_{doc_type}"}))
    monthly = monthly.merge(sub, on="date", how="left")
    monthly[f"sent_{doc_type}"] = monthly[f"sent_{doc_type}"].ffill()

# Weighted composite: MPC matters most (35%), budget (30%), governor (20%), survey (15%)
# Documents only exist from Oct 2016 (MPC established) — earlier months get 0 (neutral),
# matching the TRD's gap-handling strategy for sent_mpc.
monthly["finbert_composite"] = (
    monthly["sent_mpc"].fillna(0) * 0.35 +
    monthly["sent_budget"].fillna(0) * 0.30 +
    monthly["sent_rbi_governor"].fillna(0) * 0.20 +
    monthly["sent_econ_survey"].fillna(0) * 0.15
)
monthly["sent_mpc_available"] = monthly["sent_mpc"].notna()

merge_cols = ["date", "finbert_composite", "sent_mpc", "sent_budget",
              "sent_rbi_governor", "sent_econ_survey", "sent_mpc_available"]

# Drop old versions of these columns from master if they already exist
# (they may be flat-filled placeholders from Layer 1's initial merge)
existing_cols = [c for c in merge_cols[1:] if c in master.columns]
master_clean = master.drop(columns=existing_cols)

master_enhanced = master_clean.merge(monthly[merge_cols], on="date", how="left")
master_enhanced[["finbert_composite", "sent_mpc", "sent_budget",
                  "sent_rbi_governor", "sent_econ_survey"]] = \
    master_enhanced[["finbert_composite", "sent_mpc", "sent_budget",
                      "sent_rbi_governor", "sent_econ_survey"]].fillna(0)

# Save as a NEW file — does not overwrite extended_2000_2026.csv, so your
# already-locked Layer 1 model/training data stays untouched.
out_path = DATA_DIR / "extended_2000_2026_with_sentiment.csv"
master_enhanced.to_csv(out_path, index=False)

print(f"Enhanced dataset saved: {out_path}")
print(f"Shape: {master_enhanced.shape}")
print("\nSentiment around 2020 COVID recession:")
mask = (master_enhanced["date"] >= "2020-01-01") & (master_enhanced["date"] <= "2020-12-01")
print(master_enhanced[mask][["date", "recession_label", "finbert_composite", "sent_mpc"]].to_string(index=False))

## Summary

Layer 2 is now complete and does the real thing, not a stub:
- Downloads ~37 real RBI/Budget/Economic Survey documents into `backend/policy_docs/`
- Extracts clean text (PDF via `pdfplumber`, HTML via BeautifulSoup `<p>`/`<td>` extraction)
- Chunks by tokens (400 tokens, up to 20 chunks/doc) — respects FinBERT's 512-token limit
- Scores every chunk with FinBERT, averages into a per-document composite score
- Builds a monthly weighted sentiment composite (MPC 35%, Budget 30%, Governor 20%, Survey 15%)
- Saves an enhanced dataset with real sentiment features, as a **separate file** so your locked Layer 1 model/training isn't touched

**Note:** this doesn't automatically retrain Layer 1 with the new sentiment features —
that was intentionally left as your call, since Layer 1's GradientBoosting model
is already tuned and locked. If you want to test whether `finbert_composite`/`sent_mpc`
improve on the 10/13 test recall, re-run the Layer 1 training cells pointing at
`extended_2000_2026_with_sentiment.csv` instead, and compare.